In [ ]:
# ============================================================
# CHECK NEW TEST SET SIZE
# ============================================================
import json
from google.colab import drive
drive.mount('/content/drive')

with open('/content/drive/MyDrive/input/sentences_src.json') as f:
    src_data = json.load(f)

print(f"Total sentences: {len(src_data)}")
print(f"Sample entry: {json.dumps(src_data[0], indent=2)}")

# Estimate time at different delays
for delay in [0.0, 0.1, 0.2, 0.5]:
    est = len(src_data) * (1.0 + delay) / 60
    print(f"  delay={delay}s → ~{est:.0f} min")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total sentences: 48809
Sample entry: {
  "pair_id": "CD015746",
  "source": "Cochrane-auto 2026",
  "language": "en",
  "para_id": 0,
  "sent_id": 0,
  "complex": "We included 19 trials (17 RCTs and two cluster-RCTs)."
}
  delay=0.0s → ~813 min
  delay=0.1s → ~895 min
  delay=0.2s → ~976 min
  delay=0.5s → ~1220 min


In [ ]:
# ============================================================
# PARALLEL SUBMISSION V2 — with live progress every 10s
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'groq', '-q'])

import os, json, zipfile, time, threading
from groq import Groq
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive
drive.mount('/content/drive')

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"  # ← paste key
N_WORKERS = 10
clients   = [Groq(api_key=os.environ["GROQ_API_KEY"]) for _ in range(N_WORKERS)]

# Verify
try:
    r = clients[0].chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": "Say READY"}],
        max_tokens=5
    )
    print(f"✅ Key works: {r.choices[0].message.content}")
except Exception as e:
    print(f"❌ {e}"); raise

DATA_PATH = '/content/drive/MyDrive/input/sentences_src.json'
OUT_DIR   = '/content/drive/MyDrive/SimpleText2025/outputs'
OUT_PATH  = f'{OUT_DIR}/tokatrons_task11_LLaMA4Scout_2026.json'
ZIP_PATH  = OUT_PATH.replace('.json', '.zip')
CKPT_PATH = f'{OUT_DIR}/scout_2026_checkpoint.json'
os.makedirs(OUT_DIR, exist_ok=True)

with open(DATA_PATH) as f:
    src_data = json.load(f)
total = len(src_data)
print(f"✅ Loaded {total} sentences")

SYSTEM_PROMPT = """You are a medical text simplifier. Rewrite the sentence in plain English for a 13-year-old with no medical background.
RULES:
- Output ONLY the rewritten sentence, nothing else
- Always change the wording — never copy input unchanged
- Replace medical jargon with everyday words:
  * RCT → clinical trial, odds ratio / OR → chance of
  * heterogeneity → variation between studies
  * glucocorticoids → steroid medicines
  * confidence interval / CI → range of uncertainty
  * I2 → how different the study results were
- Keep sentences short and clear
- Target Grade 8 reading level"""

def simplify_one(args):
    i, sentence, cli = args
    for attempt in range(3):
        try:
            resp = cli.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": sentence}
                ],
                max_tokens=150,
                temperature=0.3
            )
            result = resp.choices[0].message.content.strip()
            return i, result if result else sentence
        except Exception as e:
            time.sleep(5 * (attempt + 1))
    return i, sentence

# Load checkpoint
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        ckpt = json.load(f)
    results_dict = {int(k): v for k, v in ckpt['results'].items()}
    print(f"✅ Resuming — {len(results_dict)}/{total} done")
else:
    results_dict = {}
    print("Starting fresh")

remaining = [(i, src_data[i]['complex']) for i in range(total)
             if i not in results_dict]
print(f"Remaining: {len(remaining)}\n")

# ── Background thread: saves checkpoint + prints every 10s ──
stop_monitor = threading.Event()

def monitor():
    last_count = len(results_dict)
    while not stop_monitor.is_set():
        time.sleep(10)
        current = len(results_dict)
        rate    = (current - last_count) / 10  # per second
        elapsed = time.time() - start_time
        eta     = (len(remaining) - current + len(results_dict) - (total - len(remaining))) / rate if rate > 0 else 9999
        print(f"  ⏱ {current}/{total} | +{current-last_count} in 10s "
              f"({rate:.1f}/s) | ETA: {eta/60:.0f} min")
        # Save checkpoint
        with open(CKPT_PATH, 'w') as f:
            json.dump({'results': {str(k): v for k, v in results_dict.items()}}, f)
        last_count = current

start_time = time.time()
monitor_thread = threading.Thread(target=monitor, daemon=True)
monitor_thread.start()
print("Monitor started — progress every 10s\n")

# ── Run all parallel in one shot ──
lock = threading.Lock()
args = [(i, sent, clients[j % N_WORKERS])
        for j, (i, sent) in enumerate(remaining)]

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(simplify_one, arg): arg for arg in args}
    for future in as_completed(futures):
        i, pred = future.result()
        with lock:
            results_dict[i] = pred

stop_monitor.set()
print(f"\n✅ All {len(results_dict)} predictions done!")

# Final checkpoint
with open(CKPT_PATH, 'w') as f:
    json.dump({'results': {str(k): v for k, v in results_dict.items()}}, f)

# ── Build submission ──
submission = []
for i, entry in enumerate(src_data):
    pred = results_dict.get(i, entry['complex'])
    submission.append({
        "pair_id":    str(entry['pair_id']),
        "source":     entry.get('source', 'Cochrane-auto 2026'),
        "language":   entry.get('language', 'en'),
        "para_id":    int(entry['para_id']),
        "sent_id":    int(entry['sent_id']),
        "complex":    entry['complex'],
        "prediction": pred if pred.strip() != "" else entry['complex'],
        "run_id":     "tokatrons_task11_LLaMA4Scout"
    })

with open(OUT_PATH, 'w') as f:
    json.dump(submission, f, indent=2)
with zipfile.ZipFile(ZIP_PATH, 'w') as zf:
    zf.write(OUT_PATH, arcname='tokatrons_task11_LLaMA4Scout_2026.json')

identical = sum(1 for s in submission
                if s['prediction'].strip() == s['complex'].strip())
print(f"\n✅ Submission saved: {ZIP_PATH}")
print(f"📊 Total: {len(submission)} | Changed: {len(submission)-identical} | Identical: {identical}")
print(f"\nSample:")
for entry in submission[:2]:
    print(json.dumps(entry, indent=4))

if os.path.exists(CKPT_PATH):
    os.remove(CKPT_PATH)
    print("\n✅ Checkpoint cleaned up")

  ⏱ 15028/48809 | +0 in 10s (0.0/s) | ETA: 167 min
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Key works: READY
✅ Loaded 48809 sentences
✅ Resuming — 15028/48809 done
Remaining: 33781

Monitor started — progress every 10s

  ⏱ 15047/48809 | +19 in 10s (1.9/s) | ETA: 164 min
  ⏱ 15047/48809 | +19 in 10s (1.9/s) | ETA: 164 min
  ⏱ 15047/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 15047/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 15047/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 15047/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 15057/48809 | +10 in 10s (1.0/s) | ETA: 313 min
  ⏱ 15057/48809 | +10 in 10s (1.0/s) | ETA: 313 min
  ⏱ 15057/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 15057/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 15057/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 15057/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 15067/48809 | +10 in 10s (1.0/s) | ETA: 313 min
  ⏱ 15067/48809 | +10 

KeyboardInterrupt: 

In [ ]:
# ============================================================
# PARALLEL SUBMISSION V2 — PART 2 (32001 - 40000) FIXED
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'groq', '-q'])

import os, json, zipfile, time, threading, re
from groq import Groq, RateLimitError
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter, defaultdict
from google.colab import drive

drive.mount('/content/drive')

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"
# Get your API key at https://console.groq.com/keys

N_WORKERS = 10
clients   = [Groq(api_key=os.environ["GROQ_API_KEY"], timeout=15.0) for _ in range(N_WORKERS)]

DATA_PATH = '/content/drive/MyDrive/input/sentences_src.json'
OUT_DIR   = '/content/drive/MyDrive/SimpleText2025/outputs'

# ── Separate checkpoint file for part 2 ──
OUT_PATH  = f'{OUT_DIR}/tokatrons_task11_LLaMA4Scout_2026_part2.json'
ZIP_PATH  = OUT_PATH.replace('.json', '.zip')
CKPT_PATH = f'{OUT_DIR}/scout_2026_checkpoint_part2_SEPARATE.json'   # <-- separate file
os.makedirs(OUT_DIR, exist_ok=True)

with open(DATA_PATH) as f:
    src_data = json.load(f)
total = len(src_data)

# ── Strict Target Constraints ──
START_IDX        = 32001
END_IDX          = 40000
TOTAL_RANGE_TASKS = (END_IDX - START_IDX) + 1   # Exactly 8000 items

# ── Per-language identical-rate tracking ──
# If a language's predictions are "same as source" above this threshold
# (after seeing at least MIN_LANG_SAMPLES samples), auto-skip it.
IDENTICAL_THRESHOLD = 0.85   # 85 % identical → treat whole language as copy-through
MIN_LANG_SAMPLES    = 5      # need at least this many seen before we decide

lang_stats_lock = threading.Lock()
lang_stats: dict[str, dict] = defaultdict(lambda: {"total": 0, "identical": 0, "skip": False})

def record_lang_result(lang: str, complex_text: str, prediction: str):
    """Update per-language stats and flip skip flag when threshold is crossed."""
    with lang_stats_lock:
        st = lang_stats[lang]
        if st["skip"]:
            return
        st["total"] += 1
        if prediction.strip() == complex_text.strip():
            st["identical"] += 1
        if st["total"] >= MIN_LANG_SAMPLES:
            rate = st["identical"] / st["total"]
            if rate >= IDENTICAL_THRESHOLD:
                st["skip"] = True
                print(f"\n⚡ Auto-skip activated for language '{lang}' "
                      f"(identical rate {rate:.0%} over {st['total']} samples). "
                      f"Future sentences in this language will revert to source.\n")

def should_skip_lang(lang: str) -> bool:
    with lang_stats_lock:
        return lang_stats[lang]["skip"]

# ── System Prompt ──
SYSTEM_PROMPT = """You are a medical text simplifier. Rewrite the sentence in plain English for a 13-year-old with no medical background.
RULES:
- Output ONLY the rewritten sentence, nothing else
- Always change the wording — never copy input unchanged
- Replace medical jargon with everyday words:
  * RCT → clinical trial, odds ratio / OR → chance of
  * heterogeneity → variation between studies
  * glucocorticoids → steroid medicines
  * confidence interval / CI → range of uncertainty
  * I2 → how different the study results were
- Keep sentences short and clear
- Target Grade 8 reading level"""

# ── Core simplification function (bug-fixed) ──
def simplify_one(args):
    i, sentence, lang, cli = args

    # Fast-path: language already flagged for auto-skip
    if should_skip_lang(lang):
        return i, sentence, lang, True   # True = was skipped

    for attempt in range(5):
        try:
            resp = cli.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": sentence}
                ],
                max_tokens=150,
                temperature=0.3
            )
            # BUG FIX: was resp.choices.message — must index into list
            result = resp.choices[0].message.content.strip()
            prediction = result if result else sentence

            # Record result for language-rate tracking (English included)
            record_lang_result(lang, sentence, prediction)
            return i, prediction, lang, False

        except RateLimitError as e:
            err_msg   = str(e)
            wait_time = 60
            match_ms  = re.search(r'try again in (\d+)m([\d.]+)s', err_msg)
            match_s   = re.search(r'try again in ([\d.]+)s', err_msg)
            if match_ms:
                wait_time = int(match_ms.group(1)) * 60 + float(match_ms.group(2))
            elif match_s:
                wait_time = float(match_s.group(1))
            wait_time = int(wait_time) + 5
            print(f"\n🛑 Rate Limit at item {i}. Pausing {wait_time}s …")
            time.sleep(wait_time)

        except Exception as e:
            print(f"  ⚠️  Error at item {i} attempt {attempt+1}: {e}")
            time.sleep(5 * (attempt + 1))

    # All retries exhausted → fall back to source
    return i, sentence, lang, False

# ── Load separate checkpoint (safe against corruption) ──
results_dict: dict[int, str] = {}
if os.path.exists(CKPT_PATH):
    try:
        with open(CKPT_PATH) as f:
            ckpt = json.load(f)
        results_dict = {int(k): v for k, v in ckpt.get('results', {}).items()}
        done_in_range = sum(1 for idx in results_dict if START_IDX <= idx <= END_IDX)
        print(f"✅ Checkpoint loaded ({CKPT_PATH}). Already done in range: {done_in_range}")
    except (json.JSONDecodeError, KeyError):
        print("⚠️  Checkpoint corrupted or unreadable — starting fresh for this range.")
        results_dict = {}
else:
    print(f"ℹ️  No checkpoint found at {CKPT_PATH}. Starting from scratch.")

# ── Build work queue ──
# Languages with 0 % chance of being processed by the model are skipped up-front
# (i.e. already known non-English from the dataset field)
remaining       = []
skipped_non_en  = 0

for i in range(START_IDX, min(END_IDX + 1, total)):
    if i in results_dict:
        continue
    entry = src_data[i]
    lang  = entry.get('language', 'en').lower().strip()
    text  = entry.get('complex', '')

    if lang != 'en':
        # Non-English: copy source immediately (no API call)
        results_dict[i] = text
        skipped_non_en += 1
    else:
        remaining.append((i, text, lang))

print(f"\n🎯 Processing window : indices {START_IDX} → {min(END_IDX, total)}")
print(f"⚡ Pre-skipped (non-English)     : {skipped_non_en}")
print(f"📋 English tasks for Groq API   : {len(remaining)}\n")

# ── Monitor thread (checkpoint save with lock) ──
ckpt_lock    = threading.Lock()
stop_monitor = threading.Event()

def save_checkpoint():
    with ckpt_lock:
        tmp = CKPT_PATH + '.tmp'
        with open(tmp, 'w') as f:
            json.dump({'results': {str(k): v for k, v in results_dict.items()}}, f)
        os.replace(tmp, CKPT_PATH)   # atomic replace

def monitor():
    def range_count():
        return sum(1 for idx in results_dict if START_IDX <= idx <= END_IDX)

    last = range_count()
    while not stop_monitor.is_set():
        time.sleep(10)
        current = range_count()
        rate    = (current - last) / 10
        pct     = current / TOTAL_RANGE_TASKS * 100
        print(f"  ⏱  [{pct:5.1f}%] {current}/{TOTAL_RANGE_TASKS} done | "
              f"+{current - last} in 10 s ({rate:.1f}/s)")

        # Show any newly flagged languages
        with lang_stats_lock:
            for lg, st in lang_stats.items():
                if st["skip"]:
                    print(f"       ↳ '{lg}' on auto-skip (rate {st['identical']}/{st['total']})")

        save_checkpoint()
        last = current

monitor_thread = threading.Thread(target=monitor, daemon=True)
monitor_thread.start()
print("Monitor started — saving checkpoint every 10 s\n")

# ── Parallel execution ──
data_lock = threading.Lock()
args_list = [
    (i, sent, lang, clients[j % N_WORKERS])
    for j, (i, sent, lang) in enumerate(remaining)
]

if args_list:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(simplify_one, arg): arg for arg in args_list}
        for future in as_completed(futures):
            idx, pred, lang, was_skipped = future.result()
            with data_lock:
                results_dict[idx] = pred

stop_monitor.set()
monitor_thread.join(timeout=15)

# Final checkpoint save
save_checkpoint()
print(f"\n✅ All tasks for range {START_IDX}–{END_IDX} completed!")

# ── Build submission JSON ──
submission = []
for i in range(START_IDX, min(END_IDX + 1, total)):
    entry = src_data[i]
    pred  = results_dict.get(i, entry.get('complex', ''))
    # Safety: never submit an empty prediction
    if not pred or pred.strip() == '':
        pred = entry.get('complex', '')
    submission.append({
        "pair_id":    str(entry.get('pair_id', i)),
        "source":     entry.get('source', 'Cochrane-auto 2026'),
        "language":   entry.get('language', 'en'),
        "para_id":    int(entry.get('para_id', 0)),
        "sent_id":    int(entry.get('sent_id', 0)),
        "complex":    entry.get('complex', ''),
        "prediction": pred,
        "run_id":     "tokatrons_task11_LLaMA4Scout"
    })

with open(OUT_PATH, 'w') as f:
    json.dump(submission, f, indent=2, ensure_ascii=False)
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT_PATH, arcname='tokatrons_task11_LLaMA4Scout_2026_part2.json')

identical = sum(1 for s in submission
                if s['prediction'].strip() == s['complex'].strip())
changed   = len(submission) - identical

print(f"\n📦 Submission zip : {ZIP_PATH}")
print(f"📊 Total          : {len(submission)}")
print(f"   ✏️  Changed      : {changed}")
print(f"   🔁 Identical    : {identical}")

lang_dist = Counter(s['language'] for s in submission)
print(f"\n🌐 Language distribution : {dict(lang_dist)}")

print("\n🔍 Per-language identical-rate summary:")
with lang_stats_lock:
    for lg, st in sorted(lang_stats.items()):
        rate = st['identical'] / st['total'] if st['total'] else 0
        flag = " ← AUTO-SKIPPED" if st['skip'] else ""
        print(f"   {lg:6s} | seen {st['total']:4d} | identical {st['identical']:4d} "
              f"| rate {rate:.0%}{flag}")

  ⏱  [ 95.6%] 7648/8000 done | +0 in 10 s (0.0/s)
  ⏱ 7648/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ 7648/48809 | +0 in 10s (0.0/s) | ETA: 167 min
  ⏱ Range Progress: 7648/8000 sentences | +0 in 10s (0.0/s)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ⏱ Range Progress: 7648/8000 | +0 in 10s (0.0/s)
  ⏱  [ 95.6%] 7648/8000 done | +0 in 10 s (0.0/s)
  ⏱ 7648/48809 | +0 in 10s (0.0/s) | ETA: 167 min
⚠️  Checkpoint corrupted or unreadable — starting fresh for this range.

🎯 Processing window : indices 32001 → 40000
⚡ Pre-skipped (non-English)     : 7648
📋 English tasks for Groq API   : 352

Monitor started — saving checkpoint every 10 s

  ⏱ 7679/48809 | +31 in 10s (3.1/s) | ETA: -259 min
  ⏱ Range Progress: 7679/8000 sentences | +31 in 10s (3.1/s)

🛑 Rate Limit at item 39680. Pausing 7s …

🛑 Rate Limit at item 39686. Pausing 7s …

🛑 Rate Limit at item 39681. Pausing 7s …

🛑 Rate Limit at item 39685. P

In [ ]:
# ============================================================
# STRICT POSITIONAL VALIDATOR — Part 2 (32001–40000)
# Confirms every submission row's 'complex' field exactly
# matches src_data[i]['complex'] for i = 32001..40000
# ============================================================

import json
from collections import defaultdict

OUT_DIR    = '/content/drive/MyDrive/SimpleText2025/outputs'
SUBMISSION = f'{OUT_DIR}/tokatrons_task11_LLaMA4Scout_2026_part2.json'
DATA_PATH  = '/content/drive/MyDrive/input/sentences_src.json'

START_IDX, END_IDX = 32001, 40000
EXPECTED_COUNT     = END_IDX - START_IDX + 1  # 8000

print("📂 Loading files …")
with open(SUBMISSION) as f:
    submission = json.load(f)
with open(DATA_PATH) as f:
    src_data = json.load(f)

print(f"✅ Submission rows : {len(submission)}")
print(f"✅ Source rows     : {len(src_data)}")
print()

# ── Check 1: Count ──
assert len(submission) == EXPECTED_COUNT, \
    f"❌ Expected {EXPECTED_COUNT} rows, got {len(submission)}"
print(f"✅ CHECK 1 PASSED — Row count is exactly {EXPECTED_COUNT}")

# ── Check 2: Positional source match ──
# submission[0] should correspond to src_data[32001],
# submission[1] → src_data[32002], etc.
mismatches       = []
missing_pred     = []
empty_pred       = []
pair_id_mismatches = []

for offset, rec in enumerate(submission):
    src_idx    = START_IDX + offset
    src_entry  = src_data[src_idx]

    sub_complex = rec.get('complex', '').strip()
    src_complex = src_entry.get('complex', '').strip()
    sub_pairid  = str(rec.get('pair_id', ''))
    src_pairid  = str(src_entry.get('pair_id', ''))
    prediction  = rec.get('prediction', '').strip()

    # 2a: Does 'complex' field match source exactly?
    if sub_complex != src_complex:
        mismatches.append({
            'submission_position': offset,
            'src_index':           src_idx,
            'pair_id_in_sub':      sub_pairid,
            'pair_id_in_src':      src_pairid,
            'sub_complex':         sub_complex[:120],
            'src_complex':         src_complex[:120],
        })

    # 2b: pair_id agreement
    if sub_pairid != src_pairid:
        pair_id_mismatches.append({
            'offset': offset, 'src_index': src_idx,
            'sub_pair_id': sub_pairid, 'src_pair_id': src_pairid
        })

    # 2c: Empty or missing prediction
    if not prediction:
        empty_pred.append({'src_index': src_idx, 'pair_id': sub_pairid})

# ── Report Check 2a ──
if not mismatches:
    print(f"✅ CHECK 2a PASSED — All {EXPECTED_COUNT} 'complex' fields match "
          f"src_data[{START_IDX}]..src_data[{END_IDX}] exactly")
else:
    print(f"❌ CHECK 2a FAILED — {len(mismatches)} positional mismatches found!\n")
    for m in mismatches[:10]:
        print(f"   Offset {m['submission_position']} → src_index {m['src_index']}")
        print(f"     pair_id  sub={m['pair_id_in_sub']}  src={m['pair_id_in_src']}")
        print(f"     SUB complex: {m['sub_complex']}")
        print(f"     SRC complex: {m['src_complex']}")
        print()
    if len(mismatches) > 10:
        print(f"   … and {len(mismatches)-10} more mismatches")

# ── Report Check 2b ──
if not pair_id_mismatches:
    print(f"✅ CHECK 2b PASSED — All pair_ids match between submission and source")
else:
    print(f"❌ CHECK 2b FAILED — {len(pair_id_mismatches)} pair_id mismatches!\n")
    for m in pair_id_mismatches[:10]:
        print(f"   Offset {m['offset']} → src_index {m['src_index']} | "
              f"sub={m['sub_pair_id']} vs src={m['src_pair_id']}")

# ── Report Check 2c ──
if not empty_pred:
    print(f"✅ CHECK 2c PASSED — No empty predictions")
else:
    print(f"❌ CHECK 2c FAILED — {len(empty_pred)} empty predictions found:")
    for e in empty_pred:
        print(f"   src_index {e['src_index']} | pair_id {e['pair_id']}")

# ── Check 3: Index coverage (no gaps or duplicates via pair_id) ──
seen_pairids = [str(r['pair_id']) for r in submission]
if len(seen_pairids) == len(set(seen_pairids)):
    print(f"✅ CHECK 3  PASSED — All {EXPECTED_COUNT} pair_ids are unique (no duplicates)")
else:
    dupes = [p for p in set(seen_pairids) if seen_pairids.count(p) > 1]
    print(f"❌ CHECK 3  FAILED — {len(dupes)} duplicate pair_ids: {dupes[:10]}")

# ── Check 4: Language field consistency ──
lang_mismatch = []
for offset, rec in enumerate(submission):
    src_idx  = START_IDX + offset
    sub_lang = rec.get('language','').lower().strip()
    src_lang = src_data[src_idx].get('language','').lower().strip()
    if sub_lang != src_lang:
        lang_mismatch.append((src_idx, sub_lang, src_lang))

if not lang_mismatch:
    print(f"✅ CHECK 4  PASSED — All language fields match source")
else:
    print(f"❌ CHECK 4  FAILED — {len(lang_mismatch)} language mismatches:")
    for idx, sl, rl in lang_mismatch[:5]:
        print(f"   src_index {idx} | submission='{sl}' source='{rl}'")

# ── Final verdict ──
all_passed = (not mismatches and not pair_id_mismatches
              and not empty_pred and not lang_mismatch
              and len(seen_pairids) == len(set(seen_pairids)))

print()
print("=" * 55)
if all_passed:
    print("🎉 ALL CHECKS PASSED — Safe to submit!")
    print(f"   Every one of the {EXPECTED_COUNT} rows is correctly")
    print(f"   aligned to src_data[{START_IDX}]..src_data[{END_IDX}]")
else:
    print("⚠️  SOME CHECKS FAILED — Review issues above before submitting")
print("=" * 55)

📂 Loading files …
✅ Submission rows : 8000
✅ Source rows     : 48809

✅ CHECK 1 PASSED — Row count is exactly 8000
✅ CHECK 2a PASSED — All 8000 'complex' fields match src_data[32001]..src_data[40000] exactly
✅ CHECK 2b PASSED — All pair_ids match between submission and source
✅ CHECK 2c PASSED — No empty predictions
❌ CHECK 3  FAILED — 275 duplicate pair_ids: ['10.1002/14651858.CD016031/pt', '10.1002/14651858.CD012925.pub2/ko', '10.1002/14651858.CD002946.pub2/ja', '10.1002/14651858.CD015363.pub2/pt', '10.1002/14651858.CD013342.pub2/zh_HANS', '10.1002/14651858.CD015806.pub2/fr', '10.1002/14651858.CD013848.pub2/th', '10.1002/14651858.CD014428.pub2/zh_HANS', '10.1002/14651858.CD014497.pub2/zh_HANS', '10.1002/14651858.CD013835.pub2/zh_HANS']
✅ CHECK 4  PASSED — All language fields match source

⚠️  SOME CHECKS FAILED — Review issues above before submitting


In [ ]:
# ============================================================
# BUILD PARTIAL SUBMISSION FROM CHECKPOINT (EXACTLY 16000)
# Uses the first 16000 predictions saved so far
# Language tag comes from source document (not hardcoded)
# ============================================================
import json, zipfile, os
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/input/sentences_src.json'
OUT_DIR   = '/content/drive/MyDrive/SimpleText2025/outputs'
CKPT_PATH = f'{OUT_DIR}/scout_2026_checkpoint.json'
OUT_PATH  = f'{OUT_DIR}/tokatrons_task11_LLaMA4Scout_partial.json'
ZIP_PATH  = OUT_PATH.replace('.json', '.zip')

# Load source data
with open(DATA_PATH) as f:
    src_data = json.load(f)
print(f"Total source sentences: {len(src_data)}")

# Load checkpoint safely
results_dict = {}
if not os.path.exists(CKPT_PATH):
    print("❌ No checkpoint found at:", CKPT_PATH)
else:
    try:
        with open(CKPT_PATH) as f:
            ckpt = json.load(f)

        # Verify the 'results' key exists inside the checkpoint JSON
        if ckpt and 'results' in ckpt:
            results_dict = {int(k): v for k, v in ckpt['results'].items()}
            print(f"✅ Checkpoint has {len(results_dict)} predictions")
        else:
            print("⚠️ Checkpoint file is valid JSON but missing 'results' key.")

    except json.JSONDecodeError:
        print("⚠️ Checkpoint file exists but is completely empty or corrupted.")

# Only build submission if we successfully loaded predictions
if not results_dict:
    print("❌ Cannot build submission: No predictions found in checkpoint.")
else:
    # Check language values in source
    langs = set(d.get('language', 'en') for d in src_data[:100])
    print(f"Languages in source: {langs}")

    # Sort indices and slice to keep EXACTLY the first 16000 sentences
    done_indices = sorted(results_dict.keys())[:16000]
    print(f"Building submission for EXACTLY {len(done_indices)} sentences...")

    submission = []
    for i in done_indices:
        entry = src_data[i]
        pred  = results_dict[i]
        submission.append({
            "pair_id":    str(entry['pair_id']),
            "source":     entry.get('source', 'Cochrane-auto 2026'),
            "language":   entry.get('language', 'en'),  # ✅ from source doc
            "para_id":    int(entry['para_id']),
            "sent_id":    int(entry['sent_id']),
            "complex":    entry['complex'],
            "prediction": pred if pred.strip() != "" else entry['complex'],
            "run_id":     "tokatrons_task11_LLaMA4Scout"
        })

    # Save
    os.makedirs(OUT_DIR, exist_ok=True)
    with open(OUT_PATH, 'w') as f:
        json.dump(submission, f, indent=2)
    with zipfile.ZipFile(ZIP_PATH, 'w') as zf:
        zf.write(OUT_PATH, arcname='tokatrons_task11_LLaMA4Scout_partial.json')

    identical = sum(1 for s in submission
                    if s['prediction'].strip() == s['complex'].strip())
    print(f"\n✅ Partial submission saved: {ZIP_PATH}")
    print(f"📊 Total: {len(submission)} | Changed: {len(submission)-identical} | "
          f"Identical: {identical}")
    print(f"\nSample entries:")
    for entry in submission[:2]:
        print(json.dumps(entry, indent=4))

    # Show language distribution
    from collections import Counter
    lang_dist = Counter(s['language'] for s in submission)
    print(f"\nLanguage distribution: {dict(lang_dist)}")


  ⏱ 16028/48809 | +0 in 10s (0.0/s) | ETA: 167 min
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total source sentences: 48809
✅ Checkpoint has 16028 predictions
Languages in source: {'en'}
Building submission for EXACTLY 16000 sentences...

✅ Partial submission saved: /content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_LLaMA4Scout_partial.zip
📊 Total: 16000 | Changed: 913 | Identical: 15087

Sample entries:
{
    "pair_id": "CD015746",
    "source": "Cochrane-auto 2026",
    "language": "en",
    "para_id": 0,
    "sent_id": 0,
    "complex": "We included 19 trials (17 RCTs and two cluster-RCTs).",
    "prediction": "We looked at 19 studies, including 17 clinical trials and 2 group clinical trials.",
    "run_id": "tokatrons_task11_LLaMA4Scout"
}
{
    "pair_id": "CD015746",
    "source": "Cochrane-auto 2026",
    "language": "en",
    "para_id": 0,
    "sent_id": 1,
    "complex": "The 19 tr

In [ ]:
import json
import os
from collections import Counter

DATA_PATH = '/content/drive/MyDrive/input/sentences_src.json'
OUT_DIR   = '/content/drive/MyDrive/SimpleText2025/outputs'
CKPT_PATH = f'{OUT_DIR}/scout_2026_checkpoint.json'

# Load source data
with open(DATA_PATH) as f:
    src_data = json.load(f)

# Load checkpoint
results_dict = {}
if os.path.exists(CKPT_PATH):
    try:
        with open(CKPT_PATH) as f:
            ckpt = json.load(f)
        if ckpt and 'results' in ckpt:
            results_dict = {int(k): v for k, v in ckpt['results'].items()}
    except json.JSONDecodeError:
        pass

if not results_dict:
    print("❌ No valid predictions found in checkpoint.")
else:
    # Get the exact same 16000 indices used in your submission
    done_indices = sorted(results_dict.keys())[:16000]

    # Initialize tracking structures
    lang_stats = {}

    for i in done_indices:
        entry = src_data[i]
        pred = results_dict[i]
        lang = entry.get('language', 'en')

        # Determine status
        complex_txt = entry['complex'].strip()
        pred_txt = pred.strip() if pred.strip() != "" else complex_txt
        is_identical = (pred_txt == complex_txt)

        # Initialize dictionary entry for language if not present
        if lang not in lang_stats:
            lang_stats[lang] = {'total': 0, 'identical': 0, 'changed': 0}

        lang_stats[lang]['total'] += 1
        if is_identical:
            lang_stats[lang]['identical'] += 1
        else:
            lang_stats[lang]['changed'] += 1

    # Print results in a clean table
    print("=" * 65)
    print(f"{'Language':<12} | {'Total':<10} | {'Identical':<12} | {'Changed (Different)':<15}")
    print("=" * 65)
    for lang, counts in sorted(lang_stats.items()):
        print(f"{lang:<12} | {counts['total']:<10} | {counts['identical']:<12} | {counts['changed']:<15}")
    print("=" * 65)


Language     | Total      | Identical    | Changed (Different)
de           | 67         | 65           | 2              
en           | 11099      | 10352        | 747            
es           | 4834       | 4670         | 164            


In [ ]:
import json
from collections import Counter

DATA_PATH = '/content/drive/MyDrive/input/sentences_src.json'

# Load source data
print("Loading source dataset...")
with open(DATA_PATH) as f:
    src_data = json.load(f)

total_sentences = len(src_data)
print(f"Total source sentences: {total_sentences}")

# Slice data from index 32001 to the end
remaining_data = src_data[32001:]
remaining_count = len(remaining_data)
print(f"Sentences to process (from 32001 to end): {remaining_count}\n")

# Define the step size
STEP = 4000

print("=" * 50)
print(f"{'Sentence Range':<20} | {'Language Counts'}")
print("=" * 50)

# Process in groups of 4k
for start_idx in range(0, remaining_count, STEP):
    # Calculate global indices for clear reporting
    global_start = 32001 + start_idx
    global_end = min(32001 + start_idx + STEP, total_sentences)

    # Extract the chunk
    chunk = remaining_data[start_idx : start_idx + STEP]

    # Count languages in this specific chunk
    lang_counts = Counter(d.get('language', 'en') for d in chunk)

    # Format range and print output
    range_str = f"{global_start}-{global_end}"
    print(f"{range_str:<20} | {dict(lang_counts)}")

print("=" * 50)


Loading source dataset...
Total source sentences: 48809
Sentences to process (from 32001 to end): 16808

Sentence Range       | Language Counts
32001-36001          | {'fr': 924, 'hi': 44, 'ja': 56, 'ko': 2976}
36001-40001          | {'ko': 170, 'pt': 1201, 'th': 843, 'zh_HANS': 1434, 'en': 352}
40001-44001          | {'en': 4000}
44001-48001          | {'en': 4000}
48001-48809          | {'en': 808}
